# 02_gold_weather_impact_summary

## Purpose

Create the Gold weather impact summary table.

This notebook aggregates cleaned weather Silver data into a business-facing Gold output.

## Expected output

`vattenfall_dev.analytics.gold_weather_impact_summary`

## Expected grain

One row per report day and region.

## Expected KPIs

- average temperature
- average wind speed
- maximum wind speed
- total precipitation
- weather alert indicators

## Main idea

This Gold table provides daily regional weather context that can support operational interpretation.

In [0]:
import sys
import importlib

repo_src_path = "/Workspace/Users/dirella@startsteps.org/vattenfall-week9-capstone-anna/src"

if repo_src_path not in sys.path:
    sys.path.append(repo_src_path)

import transforms.gold_aggregations as gold_aggregations
import metrics.business_metrics as business_metrics
from utils.logging_utils import log_table_operation, log_row_count

importlib.reload(gold_aggregations)
importlib.reload(business_metrics)

In [0]:
catalog = "vattenfall_dev"

source_table = f"{catalog}.refined.silver_weather"
target_table = f"{catalog}.analytics.gold_weather_impact_summary"

print("Gold table:", "Weather Impact Summary")
print("Source table:", source_table)
print("Target table:", target_table)
print("Grain: one row per report_day and region")

In [0]:
weather_df = spark.table(source_table)

source_count = weather_df.count()

log_row_count("Source Silver weather rows", source_count)

display(weather_df.limit(20))

In [0]:
gold_weather_df = (
    gold_aggregations.aggregate_weather_impact_summary(weather_df)
    .transform(business_metrics.add_weather_risk_signal)
)

gold_count = gold_weather_df.count()

log_row_count("Gold weather impact summary rows", gold_count)

display(gold_weather_df)

In [0]:
(
    gold_weather_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

log_table_operation(
    source_table=source_table,
    target_table=target_table,
    operation_name="Create Gold weather impact summary"
)

print("Written Gold table:", target_table)

In [0]:
gold_df = spark.table(target_table)

print("Rows in Gold table:", gold_df.count())
print("Columns:", gold_df.columns)

display(gold_df)

In [0]:
print("Weather risk signal distribution:")

gold_df.groupBy("weather_risk_signal").count().show()

print("Null checks for Gold grain:")

for column_name in ["report_day", "region"]:
    null_count = gold_df.filter(f"{column_name} IS NULL").count()
    print(column_name, "nulls:", null_count)

print("Duplicate grain check:")

duplicate_count = (
    gold_df
    .groupBy("report_day", "region")
    .count()
    .filter("count > 1")
    .count()
)

print("duplicate day-region rows:", duplicate_count)